# Training Notebook 2025H1 (Spark ML, Distributed)

This notebook trains Spark ML baselines on pre-engineered features and saves metrics/predictions to HDFS.

In [ ]:
import json
import urllib.request
import pandas as pd

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .appName("DemandPredictionTrainingSparkML2025H1")
    .master("spark://master:7077")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "hdfs://master:9000/spark-logs")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def cluster_util(stage_name):
    print(f"\n===== CLUSTER UTILIZATION: {stage_name} =====")
    try:
        payload = json.load(urllib.request.urlopen("http://master:8080/json/"))
        workers = payload.get("workers", [])
        print("alive workers:", payload.get("aliveworkers"))
        print("active apps :", len(payload.get("activeapps", [])))
        for w in workers:
            print("worker", w.get("id"), "cores", f"{w.get('coresused', 0)}/{w.get('cores', 0)}", "memory", f"{w.get('memoryused', 0)}/{w.get('memory', 0)}")
    except Exception as e:
        print("Could not query Spark Master JSON:", e)

cluster_util("session_started")

In [ ]:
FEATURE_PATH = "/user/data/feature_engineering/demand_prediction_2025H1_features"
OUT_BASE = "/user/data/results/demand_prediction/2025H1_sparkml"
TARGET_COL = "pickup_demand"

feature_cols = [
    "hour", "dow", "month", "is_weekend",
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12", "lag_144", "lag_1008",
    "roll_mean_6", "roll_mean_18", "roll_std_18",
]

df = spark.read.parquet(FEATURE_PATH).cache()
print("Feature rows:", df.count())
df.groupBy("split").count().show()
cluster_util("after_feature_read")

train_df = df.where(F.col("split") == "train")
test_df = df.where(F.col("split") == "test")

In [ ]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

models = [
    ("linear_regression", LinearRegression(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction", maxIter=100, regParam=0.1, elasticNetParam=0.0)),
    ("random_forest", RandomForestRegressor(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction", numTrees=180, maxDepth=12, seed=42)),
    ("gbt", GBTRegressor(featuresCol="features", labelCol=TARGET_COL, predictionCol="prediction", maxIter=120, maxDepth=6, stepSize=0.05, seed=42)),
]

mae_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")

metrics_rows = []
pred_union = None

for model_name, reg in models:
    cluster_util(f"before_train_{model_name}")
    pipeline = Pipeline(stages=[assembler, reg])
    fitted = pipeline.fit(train_df)
    pred = fitted.transform(test_df).withColumn("prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction")))

    mae_val = float(mae_eval.evaluate(pred))
    rmse_val = float(rmse_eval.evaluate(pred))
    mape_val = (
        pred.where(F.col(TARGET_COL) > 0)
        .select((F.abs(F.col(TARGET_COL) - F.col("prediction")) / F.col(TARGET_COL)).alias("ape"))
        .agg((F.avg(F.col("ape")) * 100.0).alias("mape"))
        .first()["mape"]
    )

    metrics_rows.append({
        "model": model_name,
        "MAE": mae_val,
        "RMSE": rmse_val,
        "MAPE": float(mape_val) if mape_val is not None else None,
    })

    slim_pred = pred.select(
        F.lit(model_name).alias("model"),
        "PULocationID",
        "pickup_bin_10m",
        TARGET_COL,
        "prediction",
    )
    pred_union = slim_pred if pred_union is None else pred_union.unionByName(slim_pred)
    cluster_util(f"after_train_{model_name}")

metrics_pdf = pd.DataFrame(metrics_rows).sort_values("MAPE")
display(metrics_pdf)
best_model_name = metrics_pdf.iloc[0]["model"]
print("Best model by MAPE:", best_model_name)

In [ ]:
metrics_sdf = spark.createDataFrame(metrics_rows)
metrics_sdf.write.mode("overwrite").parquet(f"{OUT_BASE}/metrics")
pred_union.write.mode("overwrite").partitionBy("model").parquet(f"{OUT_BASE}/predictions")

print("Saved outputs:")
print(f"- {OUT_BASE}/metrics")
print(f"- {OUT_BASE}/predictions")
metrics_sdf.orderBy(F.col("MAPE").asc_nulls_last()).show(truncate=False)
cluster_util("after_output_write")